###  RAG Pipeline Using LangChain

### Project Overview 

This project is a context-aware application that leverages LangChain to enable a large language model (LLM) to interact with and retrieve information from external data sources. The core of the application is a chatbot designed to answer questions about SQL notes from a Big Data course.

The Langchain framework glues together this process in this order:

📄 PDF → ✂️ Chunk → 🔢 Embeddings (1536 numbers) → 📦 Pinecone (vector database) → 🔍 Retrieve top matches → 🧠 LLM (OpenAI).

### Imports

In [2]:
import sys
python_path = sys.executable  
print("kernel python:", python_path)

# Installing pypdf and ipywidgets
!uv pip install --python "{python_path}" pypdf ipywidgets

kernel python: /usr/local/bin/python
Using Python 3.12.10 environment at: /usr/local
Resolved 21 packages in 254ms                                        
Prepared 4 packages in 211ms                                             
Installed 4 packages in 30ms.0.15                           
 + ipywidgets==8.1.7
 + jupyterlab-widgets==3.0.15
 + pypdf==6.0.0
 + widgetsnbextension==4.0.14


In [3]:
# Importing libraries + API Keys
#-------------------------------
import os
from pathlib import Path  
from dotenv import load_dotenv  

from pinecone import Pinecone, ServerlessSpec 
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore 
from langchain_community.document_loaders import PyPDFLoader  
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser  
from langchain_core.runnables.history import RunnableWithMessageHistory  
from langchain_community.chat_message_histories import ChatMessageHistory 

# Load environment variables
load_dotenv()

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") 

assert OPENAI_API_KEY, "❌ Missing OPENAI_API_KEY in .env"
assert PINECONE_API_KEY, "❌ Missing PINECONE_API_KEY in .env"

In [4]:
# API Keys
import os 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

### STEP 1: Loading and Chunking Documents

In [5]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
# loading the documents
# ---------------------

# First, we import all the PDFs (SQL Notes)
pdf_folder = Path("data/SQL PDFS")  
pdf_files = list(pdf_folder.glob("*.pdf"))
for pdf in pdf_files:
    print("-", pdf.name)

# Then we use a document loader to load all the pages in the notes
documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    documents.extend(pages)
    print(f" Loaded {len(pages)} pages from '{pdf_path.name}'")

print(f"\n Total pages loaded: {len(documents)}")

- ISOM 671 - 04 - SQL I.pdf
- ISOM 671 - 05 - SQL II.pdf
- ISOM 671 - 06 - SQL III.pdf
 Loaded 70 pages from 'ISOM 671 - 04 - SQL I.pdf'
 Loaded 72 pages from 'ISOM 671 - 05 - SQL II.pdf'
 Loaded 39 pages from 'ISOM 671 - 06 - SQL III.pdf'

 Total pages loaded: 181


In [6]:
# Step 2: Chunking the Documents
# -------------------------------

from langchain.text_splitter import RecursiveCharacterTextSplitter

# Configure chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    
    chunk_overlap=100  
)

chunks = text_splitter.split_documents(documents)
print(f"\n🔪 Split into {len(chunks)} chunks")
print("Sample chunk metadata:", chunks[0].metadata)  


🔪 Split into 244 chunks
Sample chunk metadata: {'source': 'data/SQL PDFS/ISOM 671 - 04 - SQL I.pdf', 'page': 0}


I used a RecursiveCharacterTextSplitter that splits text by \n\n (paragraphs), then \n (lines), then (words) until chunks are small enough. Smaller chunks allow better semantic search precision that are coherent (e.g a full sql query will stay together and not split).

### Step 2: Store into Vector Storage (Pinecone)

In [7]:
# Initialize Pinecone

from pinecone import Pinecone

# Initialize Pinecone 
pc = Pinecone(
    api_key=PINECONE_API_KEY
)

print(" Pinecone initialized successfully!")

 Pinecone initialized successfully!


In [8]:
## Creating a new index : This is basically a database with parameters for storing the chunked text.

# from pinecone import Pinecone,ServerlessSpec
# index_name = "sql-rag"

# if index_name not in pc.list_indexes():
#     pc.create_index(
#         name=index_name,
#         dimension=1536, # This is the length of vectors( numbers representing the embedded text)
#         metric="cosine", # The type of similarity search ( best for semantic search with meaning versus lexical that's keyword based)
#         spec=ServerlessSpec(cloud="aws", region="us-east-1") # Set cloud and region for deployment
#     )
# print(f"Created new index:{index_name}")

In [9]:
## Initial Storage of chunks in Pinecone Index

# from langchain_pinecone import PineconeVectorStore
# from langchain_openai import OpenAIEmbeddings

# # Initialize embeddings (Different embeddings determine the vector length
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# # Store in Pinecone 
# vectorstore = PineconeVectorStore.from_documents(
#     chunks,
#     embeddings,
#     index_name="sql-rag",
#     pinecone_api_key=PINECONE_API_KEY
# )

In [10]:
# print(f"\n🌲 Stored {len(chunks)} chunks in Pinecone!")

#### Upserting Records

In [11]:
# Use Existing Index to update the stored vectors

from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "sql-rag"
NAMESPACE  = ""  # default namespace 

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE
)
vectorstore.add_documents(chunks)  # upserts; auto-generates IDs
print(f"✅ Upserted {len(chunks)} chunks into '{INDEX_NAME}' (namespace='{NAMESPACE}')")


✅ Upserted 244 chunks into 'sql-rag' (namespace='')


### Step 3: Retrieval

#### Setting up the LLM Chain

In [17]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# prompt with context injected
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful SQL tutor. Use only the provided context. If unknown, say you don't know."),
    MessagesPlaceholder("chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

# 3) LLM + parser
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.6, api_key=OPENAI_API_KEY)
parser = StrOutputParser()

In [16]:
# Setting up the retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# quick probe
docs = retriever.invoke("What does GROUP BY do in SQL?")
print(f"Retrieved: {len(docs)} docs")
if docs: print(docs[0].page_content[:400])

Retrieved: 4 docs
GROUP BY
• The GROUP BY clause groups the rows of a result set based on one 
or more columns or expressions. 
• If you include aggregate functions in the SELECT clause, the 
aggregate is calculated for each group speciﬁed by the GROUP BY 
clause.
• The HAVING clause speciﬁes a search condition for a group or an 
aggregate. 
• MySQL applies this condition after it groups the rows that satisfy the s


### Step 4: Building a Conversational RAG Chatbot

In [21]:
# Step 4: Building a RAG Chain
#-------------------------------------------
# Import the memory component and Document type for formatting
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.documents import Document

# 1) Add Memory
# This memory object will store and manage the last 3 conversation turns.
memory = ConversationBufferWindowMemory(
    k=3,
    memory_key="chat_history",
    return_messages=True
)

# 2) Creating a function to format the retrieved documents
# The retriever returns a list of Document objects. This function joins their content into a single string to be used as the context.
def format_docs(docs: list[Document]) -> str:
    """Joins the page_content of retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)


# 3) Building the Conversational RAG Chain
# This chain handles conversation history.
conversational_rag_chain = (
    {
        # The retriever is invoked first with the user's question.
        # The result is formatted by format_docs and passed as "context".
        "context": lambda x: format_docs(retriever.invoke(x["question"])),

        # The 'question' is passed through directly.
        "question": lambda x: x["question"],

        # The 'chat_history' is loaded from the memory object.
        "chat_history": lambda x: memory.load_memory_variables(x).get("chat_history", [])
    }
    | prompt  # The prompt template receives context, question, and history
    | llm     # The LLM generates a response
    | StrOutputParser()  # The output is a string
)

# 4) This creates an interactive chat loop to run the chain
print("SQL Tutor is ready! Type 'exit' or 'quit' to end the session.")
print("-" * 50)

while True:
    try:
        user_question = input("You: ")
        if user_question.lower() in ["exit", "quit"]:
            print("Tutor: Goodbye!")
            break

        # Invoke the chain with the user's question.
        response = conversational_rag_chain.invoke({"question": user_question})
        print(f"\nTutor: {response}\n")

        # This saves the interaction to memory.
        memory.save_context(
            {"question": user_question},
            {"output": response}        
        )

    except (KeyboardInterrupt, EOFError):
        print("\nTutor: Session ended. Goodbye!")
        break

SQL Tutor is ready! Type 'exit' or 'quit' to end the session.
--------------------------------------------------


You:  what is a subquery?



Tutor: A subquery is a SELECT statement that is coded within another SQL statement. It can return a single value, a list of values, or a table of values. Subqueries can be used anywhere a single value, a list of values, or a table is allowed in SQL. The syntax for a subquery is the same as for a standard SELECT statement, but a subquery cannot include an ORDER BY clause.



You:  tell me about the sakila database?



Tutor: The Sakila database is a sample database provided by MySQL. It is often used for practicing and learning SQL queries. The Sakila database contains tables related to a fictional DVD rental store, such as tables for films, customers, rentals, and payments. The database schema is designed to showcase various SQL concepts and can be a useful resource for SQL learners. More information about the Sakila database can be found in the MySQL documentation at https://dev.mysql.com/doc/sakila/en/.



You:  use the sakila database to write a window function?



Tutor: To write a window function using the Sakila database, you can use the OVER() clause in your SQL query. Window functions allow you to perform calculations across a set of table rows related to the current row. Here is an example of a window function that calculates the average rental rate for films in the Sakila database:

```sql
SELECT
    film_id,
    title,
    rental_rate,
    AVG(rental_rate) OVER() AS avg_rental_rate
FROM
    film;
```

In this example, the window function AVG(rental_rate) OVER() calculates the average rental rate for all films in the `film` table, and the result is shown as `avg_rental_rate` in the output. You can customize window functions based on your specific requirements and the data available in the Sakila database.



You:  exit


Tutor: Goodbye!
